##### Import the required modules and configure the system path to locate them

In [16]:
import sys
sys.path.append("../../utils")
import networkx
import yaml
import os
import pandas as pd
import astra_sim_sdk.astra_sim_sdk as astra_sim_kit
from common import FileFolderUtils
from astra_sim import AstraSim, Collective, NetworkBackend
from infragraph.blueprints.devices.nvidia.dgx import NvidiaDGX
from infragraph.blueprints.fabrics.single_tier_fabric import SingleTierFabric
from infragraph.infragraph_service import InfraGraphService

##### Call the AstraSim client helper with the server endpoint and tag to connect to the ASTRA-sim gRPC server, initialize the SDK, and create a tagged folder for configs, results, and logs

In [17]:
astra = AstraSim(server_endpoint = "172.17.0.2:8989", tag = "ns3_single_tier_with_dgx")

Successfully connected to gRPC server at 172.17.0.2:8989


##### Create a single tier rack device with two Nvidia DGX and a single switch using infragraph device, fabric blueprint

In [18]:
dgx_count = 2
server = NvidiaDGX()
infrastructure = SingleTierFabric(server, dgx_count)
astra.configuration.infragraph.infrastructure.deserialize(infrastructure.serialize())

##### Initialize the Infragraph service, display the fabric topology, and retrieve/set the total number of NPUs to generate the collective

In [19]:
service = InfraGraphService()
service.set_graph(infrastructure)

g = service.get_networkx_graph()
print(networkx.write_network_text(g, vertical_chains=True))

total_npus = service.get_component(server, "xpu").count * dgx_count
print(total_npus)

╙── dgx_h100.0.pciesw.1
    │
    dgx_h100.0.cpu.1
    ├── dgx_h100.0.cpu.0
    │   ├── dgx_h100.0.pciesl.0
    │   │   ├── dgx_h100.0.xpu.0
    │   │   │   ├── dgx_h100.0.nvsw.0
    │   │   │   │   ├── dgx_h100.0.pciesw.2 ─ dgx_h100.0.cpu.0
    │   │   │   │   │   ├── dgx_h100.0.nvsw.1 ─ dgx_h100.0.xpu.0
    │   │   │   │   │   │   ├── dgx_h100.0.xpu.1 ─ dgx_h100.0.nvsw.0
    │   │   │   │   │   │   │   ├── dgx_h100.0.pciesl.1 ─ dgx_h100.0.cpu.0
    │   │   │   │   │   │   │   │   │
    │   │   │   │   │   │   │   │   dgx_h100.0.cx7.1
    │   │   │   │   │   │   │   │   │
    │   │   │   │   │   │   │   │   switch.0.port.1
    │   │   │   │   │   │   │   │   │
    │   │   │   │   │   │   │   │   switch.0.asic.0
    │   │   │   │   │   │   │   │   ├── switch.0.port.0
    │   │   │   │   │   │   │   │   │   │
    │   │   │   │   │   │   │   │   │   dgx_h100.0.cx7.0 ─ dgx_h100.0.pciesl.0
    │   │   │   │   │   │   │   │   ├── switch.0.port.2
    │   │   │   │   │   │   │   │   │   │
   

##### Generate workload execution traces for each rank and set the required data size for AstraSim configuration

In [20]:
astra.configuration.common_config.workload = astra.generate_collective(collective=Collective.ALLREDUCE, coll_size= 8 *1024*1024, npu_range=[0, total_npus])

All contents of the folder /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_dgx/configuration/workload have been deleted.
Generated 16 et in /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_dgx/configuration/workload


##### Configure ASTRA-sim system config

In [21]:
astra.configuration.common_config.system.scheduling_policy = astra.configuration.common_config.system.LIFO
astra.configuration.common_config.system.endpoint_delay = 10
astra.configuration.common_config.system.active_chunks_per_dimension = 1
astra.configuration.common_config.system.all_gather_implementation = [astra.configuration.common_config.system.RING]
astra.configuration.common_config.system.all_to_all_implementation = [astra.configuration.common_config.system.DIRECT]
astra.configuration.common_config.system.all_reduce_implementation = [astra.configuration.common_config.system.ONERING]
astra.configuration.common_config.system.collective_optimization = astra.configuration.common_config.system.LOCALBWAWARE
astra.configuration.common_config.system.local_mem_bw = 1600

##### Configure ASTRA-sim remote memory configuration

In [22]:
astra.configuration.common_config.remote_memory.memory_type = astra.configuration.common_config.remote_memory.NO_MEMORY_EXPANSION
print(astra.configuration.common_config.remote_memory)

memory_type: NO_MEMORY_EXPANSION
remote_mem_bw: 0
remote_mem_latency: 0



##### Configure the selected network backend and the topology (infragraph or nc_topology)

In [23]:
astra.configuration.network_backend.choice = astra.configuration.network_backend.NS3
astra.configuration.network_backend.ns3.topology.choice = astra.configuration.network_backend.ns3.topology.INFRAGRAPH
astra.configuration.network_backend.ns3.network.packet_payload_size = int(8192)

##### Adding ns3 trace and logical dimension 

In [24]:
astra.configuration.network_backend.ns3.logical_topology.logical_dimensions = [total_npus]
astra.configuration.network_backend.ns3.trace.trace_ids = []
for i in range(0, total_npus):
    astra.configuration.network_backend.ns3.trace.trace_ids.append(i)

##### Adding ASTRA-sim - Infragraph specific annotation

In [25]:
host_device_spec = astra_sim_kit.AnnotationDeviceSpecifications()
host_device_spec.device_bandwidth_gbps = 100
host_device_spec.device_latency_ms = 0.05
host_device_spec.device_name = server.name
host_device_spec.device_type = "host"
astra.configuration.infragraph.annotations.device_specifications.append(host_device_spec)

switch_device_spec = astra_sim_kit.AnnotationDeviceSpecifications()
switch_device_spec.device_bandwidth_gbps = 100
switch_device_spec.device_latency_ms = 0.05
switch_device_spec.device_name = "switch"
switch_device_spec.device_type = "switch"
astra.configuration.infragraph.annotations.device_specifications.append(
    switch_device_spec
)

##### Configure ASTRA-sim cmd parameters

In [26]:
astra.configuration.common_config.cmd_parameters.comm_scale = 1
astra.configuration.common_config.cmd_parameters.injection_scale = 1
astra.configuration.common_config.cmd_parameters.rendezvous_protocol = False

#### Start the simulation by specifying the network backend

In [27]:
astra.run_simulation(NetworkBackend.NS3)

Generating Configuration ZIP now
output_path: /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_dgx/config.zip
folder_path: /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_dgx/configuration/workload/..


pack_zip complete
message: 'Configuration applied successfully. warnings: Unable to generate communicator
  group message from schema - communicator group configuration empty'

message: Simulation started successfully

astra-sim server Status: running
astra-sim server Status: running
Transferring Files from ASTRA-sim server
All files downloaded Successfully
Translating Metrics...
Generated fct.csv at:  /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_dgx/output/fct.csv
Generated: flow_stats.csv at:  /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_dgx/output/flow_stats.csv
All metrics translated successfully
Simulation completed


/workspaces/astra_sim_service/client-scripts/notebooks/infragraph/../../utils/common.py:274: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(
/workspaces/astra_sim_service/client-scripts/notebooks/infragraph/../../utils/common.py:237: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(


##### Download all the configurations as a zip

In [28]:
astra.download_configuration()

Downloaded all configuration in /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_dgx/server_configuration.zip


##### Read output files

In [29]:
import pandas as pd
import os
from common import FileFolderUtils
df = pd.read_csv(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR, "fct.csv"))
df.head()

,Source Hex ip,Destination Hex ip,Source Port,Destination Port,Data size (B),Start Time,FCT,Standalone FCT
0,0b000001,0b000101,10000,100,524288,10,43026,43498
1,0b000101,0b000201,10000,100,524288,10,43026,43498
2,0b000401,0b000501,10000,100,524288,10,43026,43498
3,0b000501,0b000601,10000,100,524288,10,43026,43498
4,0b000901,0b000a01,10000,100,524288,10,43026,43498


##### Save infragraph as a yaml

In [30]:
with open(os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR,"../infrastructure","ns3_single_tier_with_dgx.yaml"),"w") as f:
    data = infrastructure.serialize("dict")
    yaml.dump(data, f, default_flow_style=False, indent=4)

print("saved yaml to:", os.path.join(FileFolderUtils.get_instance().OUTPUT_DIR,"..","ns3_single_tier_with_dgx.yaml"))

saved yaml to: /workspaces/astra_sim_service/client-scripts/utils/../trial/ns3_single_tier_with_dgx/output/../ns3_single_tier_with_dgx.yaml
